## 1. Loading the data

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

def load_any(path_stem: str):
    parquet_path = Path(f"{path_stem}.parquet")
    csv_path = Path(f"{path_stem}.csv")
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    return None

def load_concat(path_stems):
    frames = []
    for stem in path_stems:
        df = load_any(stem)
        if df is not None:
            frames.append(df)
    if not frames:
        raise FileNotFoundError(f"No data found for stems: {path_stems}")
    return pd.concat(frames, axis=0, ignore_index=True)

player_roles_master_plus= load_concat([
    "data_raw/player_roles1",
    "data_raw/player_roles2",
],)
trajectories_master_plus= load_concat([
    "data_raw/trajectories1",
    "data_raw/trajectories2",
],)

player_roles_silver = load_concat([
    "data_raw_silver/player_roles",
    "data_raw_silver/player_roles2",
    "data_raw_silver/player_roles3"
],)
trajectories_silver = load_concat([
    "data_raw_silver/trajectories",
    "data_raw_silver/trajectories2",
    "data_raw_silver/trajectories3",
],)

print(f"Master+ roles shape: {player_roles_master_plus.shape}")
print(f"Master+ trajectories shape: {trajectories_master_plus.shape}")
print(f"Silver roles shape: {player_roles_silver.shape}")
print(f"Silver trajectories shape: {trajectories_silver.shape}")

Master+ roles shape: (487950, 4)
Master+ trajectories shape: (482260, 4)
Silver roles shape: (308990, 4)
Silver trajectories shape: (308950, 4)


## 2. Preview of data and merging roles with trajectories

### 2.1 Trajectories

In [17]:
print(f"Master+ trajectories length: {trajectories_master_plus.shape}")
print(f"Silver trajectories length: {trajectories_silver.shape}")
print(trajectories_master_plus.head())
print(trajectories_silver.head())

Master+ trajectories length: (482260, 4)
Silver trajectories length: (308950, 4)
                                               puuid         match_id  team  \
0  mCG4W3ohaS1yeYuEIpAiXCgHPyOb_GsrpP_Yw6y7Y-88SI...  EUW1_7684872966     0   
1  QfvaTyT_3Ez0jaMf0_QzLaOqAwRGPKGg179xIKLtNoFoof...  EUW1_7684872966     0   
2  nJgPqmoqt_bWlQgnH9U3XDORpE9zbhgTz8HzEXRChAmH8u...  EUW1_7684872966     0   
3  D29BAybAOJFm1tgl1K_r9BOC_vHCgBNDPH6Bnjl628rDbh...  EUW1_7684872966     0   
4  3XpUlB8N3-_zm4qB0JykZOtTsETI3GBCxXKe60EAUDNYrW...  EUW1_7684872966     0   

                                           positions  
0  [[603, 611], [1032, 12131], [1401, 11087], [82...  
1  [[662, 285], [3101, 8063], [6883, 5349], [1059...  
2  [[362, 135], [7572, 7012], [7244, 7647], [8182...  
3  [[130, 401], [10633, 1241], [10709, 1448], [11...  
4  [[298, 675], [11464, 1854], [10734, 1746], [11...  
                                               puuid         match_id  team  \
0  sGOmDe7wdqygTgIwZKibZHuHJQ9pDKk5

### 2.2 Roles

In [18]:
print(f"Master+ roles length: {player_roles_master_plus.shape}")
print(f"Silver roles length: {player_roles_silver.shape}")
print(player_roles_master_plus.head())
print(player_roles_silver.head())

Master+ roles length: (487950, 4)
Silver roles length: (308990, 4)
                                               puuid         match_id  \
0  F77cxzk_zBZefmsFG-XopOhlcobFU0xIgTLxTYbwJ4TYAb...  EUW1_7682224814   
1  Elayv1cr5-cfcVkACT6tZmuDloCiNJJ1TVXxBIr6rUE0w7...  EUW1_7682224814   
2  3MuZ03offdBavQ8mRk4iAeQeDC1s25d-2cbcIQ5ceZ4vFN...  EUW1_7682224814   
3  HmSVRNlLrEwGC9ujRkJcGyskPkBqx6ahqogJbIIvZd7ZET...  EUW1_7682224814   
4  GKYhuj2C0zbsQcEpEhlw9d5kBGcPo0U5O2TL4-Atvvt13E...  EUW1_7682224814   

      role champion  
0      TOP    Fiora  
1   JUNGLE   Graves  
2   MIDDLE     Azir  
3   BOTTOM   Ezreal  
4  UTILITY     Rell  
                                               puuid         match_id  \
0  yVLgpmFydr5wI1npWSipq2FAXX-5Z2nY8MHNnyvbYF2Bfe...  EUW1_7693895387   
1  EuABY9-PHH2MssnUiVeAldxAcDA7lT8aDcHQ0NLxyUC8Gp...  EUW1_7693895387   
2  pfJPencANLopgYvvLFRPAJE8axYt7UxHMEvquVmTYL-DvZ...  EUW1_7693895387   
3  HJjROk8_joCwHChiSr1Zw6hhWkGQ2X5vGbzatxHycgIHo_...  EUW1_7693895387 

### 2.3 Merging Trajectories and Roles on puuid and match_id

In [19]:
merged_master_plus = trajectories_master_plus.merge(
    player_roles_master_plus,
    on=["puuid", "match_id"],
    how="left"
 )
merged_silver = trajectories_silver.merge(
    player_roles_silver,
    on=["puuid", "match_id"],
    how="left"
 )
print(f"Master+ merged length: {len(merged_master_plus)}")
print(f"Silver merged length: {len(merged_silver)}")
print(merged_master_plus.head())
print(merged_silver.head())

Master+ merged length: 482260
Silver merged length: 308950
                                               puuid         match_id  team  \
0  mCG4W3ohaS1yeYuEIpAiXCgHPyOb_GsrpP_Yw6y7Y-88SI...  EUW1_7684872966     0   
1  QfvaTyT_3Ez0jaMf0_QzLaOqAwRGPKGg179xIKLtNoFoof...  EUW1_7684872966     0   
2  nJgPqmoqt_bWlQgnH9U3XDORpE9zbhgTz8HzEXRChAmH8u...  EUW1_7684872966     0   
3  D29BAybAOJFm1tgl1K_r9BOC_vHCgBNDPH6Bnjl628rDbh...  EUW1_7684872966     0   
4  3XpUlB8N3-_zm4qB0JykZOtTsETI3GBCxXKe60EAUDNYrW...  EUW1_7684872966     0   

                                           positions     role champion  
0  [[603, 611], [1032, 12131], [1401, 11087], [82...      TOP     Ornn  
1  [[662, 285], [3101, 8063], [6883, 5349], [1059...   JUNGLE    Diana  
2  [[362, 135], [7572, 7012], [7244, 7647], [8182...   MIDDLE  Taliyah  
3  [[130, 401], [10633, 1241], [10709, 1448], [11...   BOTTOM    Yasuo  
4  [[298, 675], [11464, 1854], [10734, 1746], [11...  UTILITY    Leona  
                            

### 2.4 Flattening and breaking up trajectories

In [20]:
merged_master_plus["traj_len"] = merged_master_plus["positions"].apply(len)
merged_silver["traj_len"] = merged_silver["positions"].apply(len)

max_len_master_plus = merged_master_plus["traj_len"].max()
max_len_silver = merged_silver["traj_len"].max()
print("Longest game length (Master+):", max_len_master_plus)
print("Longest game length (Silver):", max_len_silver)

Longest game length (Master+): 60
Longest game length (Silver): 77


In [21]:
def normalize_positions(pos):
    """
    Converts: array([array([x,y]), array([x,y]), ...], dtype=object) -> array([[x,y], [x,y], ...])
    """
    return np.vstack(pos)

def expand_positions(pos_list, max_steps):
    out = np.full((max_steps, 2), np.nan)
    n = min(len(pos_list), max_steps)
    out[:n] = pos_list[:n]
    return out.flatten()

def flatten_positions(df, max_len):
    df = df.copy()
    df["positions_xy"] = df["positions"].apply(normalize_positions)
    expanded = np.vstack(
        df["positions_xy"].apply(lambda p: expand_positions(p, max_len))
    )
    columns = []
    for i in range(max_len):
        columns.extend([f"x{i}", f"y{i}"])
    positions_df = pd.DataFrame(expanded, columns=columns)
    return pd.concat(
        [df.drop(columns=["positions", "positions_xy"]), positions_df],
        axis=1
    )

final_df_master_plus = flatten_positions(merged_master_plus, max_len_master_plus)
final_df_silver = flatten_positions(merged_silver, max_len_silver)

print(final_df_master_plus[["x0", "y0", "x1", "y1"]].head())
print(final_df_silver[["x0", "y0", "x1", "y1"]].head())

      x0     y0       x1       y1
0  603.0  611.0   1032.0  12131.0
1  662.0  285.0   3101.0   8063.0
2  362.0  135.0   7572.0   7012.0
3  130.0  401.0  10633.0   1241.0
4  298.0  675.0  11464.0   1854.0
      x0     y0       x1       y1
0  603.0  611.0   1853.0  12112.0
1  662.0  285.0   7083.0   5174.0
2  362.0  135.0   6954.0   7417.0
3  130.0  401.0  13582.0   2290.0
4  298.0  675.0  13483.0   2711.0


## 3. Data errors, missing labels and outliers

### 3.1 Searching for errors/NaN in labels

We are looking for:
- NaN roles
- Not assigned roles which are empty but not NaN
- Duplicated which might have been created during merging

(if you have any more ideas, write them here and implement them)

In [22]:
VALID_ROLES = {"TOP", "JUNGLE", "MIDDLE", "BOTTOM", "UTILITY"}
invalid_roles_master_plus = final_df_master_plus[~final_df_master_plus["role"].isin(VALID_ROLES)]
invalid_roles_silver = final_df_silver[~final_df_silver["role"].isin(VALID_ROLES)]

print(f"Master+ - Count of nan labels: {final_df_master_plus['role'].isna().sum()}")
print(f"Master+ - Count of glitched roles: {len(invalid_roles_master_plus)}")
print(f"Master+ - Number of duplicate errors: {player_roles_master_plus.duplicated(subset=['puuid', 'match_id']).sum()}")

print(f"Silver - Count of nan labels: {final_df_silver['role'].isna().sum()}")
print(f"Silver - Count of glitched roles: {len(invalid_roles_silver)}")
print(f"Silver - Number of duplicate errors: {player_roles_silver.duplicated(subset=['puuid', 'match_id']).sum()}")

Master+ - Count of nan labels: 0
Master+ - Count of glitched roles: 72
Master+ - Number of duplicate errors: 0
Silver - Count of nan labels: 0
Silver - Count of glitched roles: 226
Master+ - Number of duplicate errors: 0
Silver - Count of nan labels: 0
Silver - Count of glitched roles: 226
Silver - Number of duplicate errors: 0
Silver - Number of duplicate errors: 0


We can either:

1. Delete them

   **Pros:**
   - No errors
   - We might need it later for some analysis (not sure)

   **Cons:**
   - Losing data (but it’s only 72 rows out of 480k)

In [23]:
data_deleted_master_plus = final_df_master_plus.drop(invalid_roles_master_plus.index)
data_deleted_silver = final_df_silver.drop(invalid_roles_silver.index)
print(data_deleted_master_plus.shape)
print(data_deleted_silver.shape)

(482188, 126)
(308724, 160)


2. Fill the role based on the majority of roles the champion is played on 

   **Pros:** 
   - Really good for some champions like kha'zix, which are played on one role only. 
   
   **Cons:**
   - Bad for champions which are played frequently on 2 roles or more.

In [24]:
champion_role_prior_master_plus = (
    final_df_master_plus[final_df_master_plus["role"].isin(VALID_ROLES)]
    .groupby(["champion", "role"])
    .size()
    .reset_index(name="count")
    .sort_values(["champion", "count"], ascending=[True, False])
    .drop_duplicates("champion")
    .set_index("champion")["role"]
 )
champion_role_prior_silver = (
    final_df_silver[final_df_silver["role"].isin(VALID_ROLES)]
    .groupby(["champion", "role"])
    .size()
    .reset_index(name="count")
    .sort_values(["champion", "count"], ascending=[True, False])
    .drop_duplicates("champion")
    .set_index("champion")["role"]
 )

invalid_roles_master_plus.loc[:, ["role_champion_majority"]] = (
    invalid_roles_master_plus["champion"].map(champion_role_prior_master_plus)
 )
invalid_roles_silver.loc[:, ["role_champion_majority"]] = (
    invalid_roles_silver["champion"].map(champion_role_prior_silver)
 )

print(invalid_roles_master_plus.loc[:, ["puuid", "match_id", "champion", "role_champion_majority"]].head())
print(invalid_roles_silver.loc[:, ["puuid", "match_id", "champion", "role_champion_majority"]].head())

                                                   puuid         match_id  \
2441   S-PwB_dwIj_BLVb9OCp9VrD9_A_-6gB9r0UrHeKGmMuTJK...   NA1_5462008265   
2798   IFykstMttbxyx1lKPlpiROJijBc0ZDIoVfJVoW8uD91DQc...  EUW1_7684088114   
9304   12p2wtYP2BN5lsJepD5MCh4GpZYP5GB6knYdnGOXv7vXX_...    KR_8027299858   
33810  s5xnoUd1sI5tpwn80u_Jgkflqqp_E-i3ie2csj-FVEFKre...  EUW1_7684307377   
38048  Yahx_bxKheeyo1MNBUVWvTI-de5QS9A8FB5kt8dS08fa2R...  EUW1_7684953000   

       champion role_champion_majority  
2441     Khazix                 JUNGLE  
2798   Tristana                 BOTTOM  
9304       Bard                UTILITY  
33810      Sett                    TOP  
38048    Twitch                 BOTTOM  
                                                   puuid         match_id  \
166    SIRbkKaBPMx72mxB2Wmb1EongDBQuH4dHNWSvJLyihkP3Y...  EUW1_7703382849   
1037   WO8Bf4fCTrHHkrVilt_UjP1U9PePmI8H-bPYWDgYQcyYiQ...  EUW1_7683297769   
9474   60Lj00VEVi-iRqlosZ6GvcXNpIwqxIUe8r3RS7CnlePP3Z...  EU

In [25]:
data_majority_master_plus = final_df_master_plus.copy()
data_majority_master_plus.loc[
    invalid_roles_master_plus.index,
    "role"
 ] = invalid_roles_master_plus["role_champion_majority"].values

data_majority_silver = final_df_silver.copy()
data_majority_silver.loc[
    invalid_roles_silver.index,
    "role"
 ] = invalid_roles_silver["role_champion_majority"].values


3. Fill the role by looking which Role isnt in the team 

   **Pros:**
   - 100% accurate when only one role from a given team in a match is missing.
   
   **Cons:** 
   - Might backfire if there is more than 1 role missing in on match.
   - Only works if riot's predictions are correct.

In [26]:
def infer_missing_role(match_id, team, team_roles):
    roles_present = team_roles.get((match_id, team), set())
    missing = VALID_ROLES - roles_present
    return next(iter(missing)) if len(missing) == 1 else None

team_roles_master_plus = (
    final_df_master_plus[final_df_master_plus["role"].isin(VALID_ROLES)]
    .groupby(["match_id", "team"])["role"]
    .apply(set)
 )
team_roles_silver = (
    final_df_silver[final_df_silver["role"].isin(VALID_ROLES)]
    .groupby(["match_id", "team"])["role"]
    .apply(set)
 )

invalid_roles_master_plus.loc[:, "role_team_missing"] = invalid_roles_master_plus.apply(
    lambda row: infer_missing_role(row["match_id"], row["team"], team_roles_master_plus),
    axis=1
 )
invalid_roles_silver.loc[:, "role_team_missing"] = invalid_roles_silver.apply(
    lambda row: infer_missing_role(row["match_id"], row["team"], team_roles_silver),
    axis=1
 )

print(invalid_roles_master_plus.loc[:, ["puuid", "match_id", "champion", "role_team_missing"]].head())
print(invalid_roles_silver.loc[:, ["puuid", "match_id", "champion", "role_team_missing"]].head())

                                                   puuid         match_id  \
2441   S-PwB_dwIj_BLVb9OCp9VrD9_A_-6gB9r0UrHeKGmMuTJK...   NA1_5462008265   
2798   IFykstMttbxyx1lKPlpiROJijBc0ZDIoVfJVoW8uD91DQc...  EUW1_7684088114   
9304   12p2wtYP2BN5lsJepD5MCh4GpZYP5GB6knYdnGOXv7vXX_...    KR_8027299858   
33810  s5xnoUd1sI5tpwn80u_Jgkflqqp_E-i3ie2csj-FVEFKre...  EUW1_7684307377   
38048  Yahx_bxKheeyo1MNBUVWvTI-de5QS9A8FB5kt8dS08fa2R...  EUW1_7684953000   

       champion role_team_missing  
2441     Khazix            JUNGLE  
2798   Tristana            BOTTOM  
9304       Bard           UTILITY  
33810      Sett               TOP  
38048    Twitch            BOTTOM  
                                                   puuid         match_id  \
166    SIRbkKaBPMx72mxB2Wmb1EongDBQuH4dHNWSvJLyihkP3Y...  EUW1_7703382849   
1037   WO8Bf4fCTrHHkrVilt_UjP1U9PePmI8H-bPYWDgYQcyYiQ...  EUW1_7683297769   
9474   60Lj00VEVi-iRqlosZ6GvcXNpIwqxIUe8r3RS7CnlePP3Z...  EUN1_3897805740   
9674   m_uCA9

C:\Users\jhaft\AppData\Local\Temp\ipykernel_21624\2842174073.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  invalid_roles_master_plus.loc[:, "role_team_missing"] = invalid_roles_master_plus.apply(
C:\Users\jhaft\AppData\Local\Temp\ipykernel_21624\2842174073.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  invalid_roles_silver.loc[:, "role_team_missing"] = invalid_roles_silver.apply(


In [28]:
data_team_master_plus = final_df_master_plus.copy()
data_team_master_plus.loc[
    invalid_roles_master_plus.index,
    "role"
 ] = invalid_roles_master_plus["role_team_missing"].values

data_team_silver = final_df_silver.copy()
data_team_silver.loc[
    invalid_roles_silver.index,
    "role"
 ] = invalid_roles_silver["role_team_missing"].values


### 3.2 Comapring method 2 and 3

In [34]:
print(f"Master+ - Roles filled by majority voting: {invalid_roles_master_plus['role_champion_majority'].notna().sum()}")
print(f"Master+ - Roles filled by Team composition analisys: {invalid_roles_master_plus['role_team_missing'].notna().sum()}")

disagreement_master_plus = invalid_roles_master_plus[
    invalid_roles_master_plus['role_champion_majority'] !=
    invalid_roles_master_plus['role_team_missing']
 ]

print(f"Master+ - Number of confliciting fillings: {len(disagreement_master_plus)}")
print(disagreement_master_plus.loc[:, ["puuid", "match_id", "champion", "role_team_missing", "role_champion_majority"]].head())

print(f"\n\nSilver - Roles filled by majority voting: {invalid_roles_silver['role_champion_majority'].notna().sum()}")
print(f"Silver - Roles filled by Team composition analisys: {invalid_roles_silver['role_team_missing'].notna().sum()}")

disagreement_silver = invalid_roles_silver[
    invalid_roles_silver['role_champion_majority'] !=
    invalid_roles_silver['role_team_missing']
 ]

print(f"Silver - Number of confliciting fillings: {len(disagreement_silver)}")
print(disagreement_silver.loc[:, ["puuid", "match_id", "champion", "role_team_missing", "role_champion_majority"]].head())

Master+ - Roles filled by majority voting: 72
Master+ - Roles filled by Team composition analisys: 72
Master+ - Number of confliciting fillings: 13
                                                    puuid         match_id  \
54835   oWGCPGCAlXqhbYyIH6PfC274TvAy-QWYTg3bdTPTKVwvuy...    KR_8027055351   
150209  f36h-i5xNhcWRlgqL2Ed-ZGjZNrYEbc2pBj3eZwCgR4FBf...  EUW1_7683614522   
181233  KLZz5yBwzSW66I3oO4X_ADpYjk5KZLfDf5wAMunJAi-FS-...  EUW1_7684386410   
230437  NJrYjtWmR3xnZOQXC5N2Te39xRuGo2XzUF1BbVwK8H5VSo...    KR_8027180300   
234082  ag3_xjhw9zYEjCNwMa7XeyvY4ayjf2QUhEpSIVndrUpW4R...    KR_8026846303   

        champion role_team_missing role_champion_majority  
54835    XinZhao               TOP                 JUNGLE  
150209       Mel           UTILITY                 MIDDLE  
181233  Aphelios           UTILITY                 BOTTOM  
230437       Zed            MIDDLE                 JUNGLE  
234082     Corki            MIDDLE                 BOTTOM  


Silver - Roles filled

As i turns out Team composition analysis is a great idea, but some of the roles in the matches are **labeled incorrectly by Riot's predictor**. Which results in silly fillings such as Yunara UTILITY, as Blitzcrank was assigned BOT this game, while the champion role predictor assigns Yunara correctly as BOT.
But due to the con we talked about champions like Zed, who can be played on multiple roles, are assigned to the majority role, while it is already filled by another jungler in that match.

In [ ]:
confusion_table_master_plus = pd.crosstab(
    disagreement_master_plus["role_team_missing"],
    disagreement_master_plus["role_champion_majority"],
    rownames=["Team-missing role"],
    colnames=["Champion-majority role"]
)
confusion_table_master_plus

Champion-majority role,BOTTOM,JUNGLE,MIDDLE,TOP
Team-missing role,,,,
BOTTOM,0,0,2,0
JUNGLE,0,0,0,1
MIDDLE,1,1,0,1
TOP,1,2,0,0
UTILITY,3,0,1,0


In [37]:
confusion_table_silver = pd.crosstab(
    disagreement_silver["role_team_missing"],
    disagreement_silver["role_champion_majority"],
    rownames=["Team-missing role"],
    colnames=["Champion-majority role"]
)
confusion_table_silver

Champion-majority role,BOTTOM,JUNGLE,MIDDLE,TOP,UTILITY
Team-missing role,,,,,
BOTTOM,0,0,3,0,0
JUNGLE,0,0,2,6,0
MIDDLE,0,3,0,4,4
TOP,3,7,9,0,0
UTILITY,10,0,2,1,0


### 3.3 See how many riot labeling errors we can detect in the data(outliers)  ???

## x. Saving the data

In [ ]:
from pathlib import Path

Path("data_processed_master_plus").mkdir(parents=True, exist_ok=True)
Path("data_processed_silver").mkdir(parents=True, exist_ok=True)

data_deleted_master_plus.to_parquet("data_processed_master_plus/data_deleted.parquet", index=False)
data_majority_master_plus.to_parquet("data_processed_master_plus/data_majority.parquet", index=False)
data_team_master_plus.to_parquet("data_processed_master_plus/data_team.parquet", index=False)

data_deleted_silver.to_parquet("data_processed_silver/data_deleted.parquet", index=False)
data_majority_silver.to_parquet("data_processed_silver/data_majority.parquet", index=False)
data_team_silver.to_parquet("data_processed_silver/data_team.parquet", index=False)